In [3]:
import pandas as pd
import numpy as np

In [14]:
class NaiveBayesClassifier:
    def __init__(self):
        self.spam_prob = 0.5
        self.notSpam_prob = 0.5
        self.data = None
        self.spam_likelihoods = {}
        self.nonspam_likelihoods = {}
        
    def loadData(self, data):
        self.data = data

    def compute_initial_probs(self):
        data = self.data
        nrecords = len(data)
        
        counts = data['label'].value_counts()
        spam_counts = counts[1]
        not_spam_counts = counts[0]
        
        total = spam_counts + not_spam_counts
        
        self.spam_prob = spam_counts / total
        self.notSpam_prob = not_spam_counts / total
    
        return self.spam_prob, self.notSpam_prob
    
    def compute_histograms(self):
        df = self.data
        
        spam_tokens_freq = {}
        not_spam_tokens_freq = {}
        total_spam_tokens = 0
        total_nonspam_tokens = 0
        
        for _, row in df[['email', 'label']].dropna(subset=['email']).iterrows():
            email = row['email']
            label = row['label']
            email_str = str(email)
            
            for token in email_str.split():
                if label == 0:
                    total_nonspam_tokens += 1
                    not_spam_tokens_freq[token] = not_spam_tokens_freq.get(token, 0) + 1
                else:
                    total_spam_tokens += 1
                    spam_tokens_freq[token] = spam_tokens_freq.get(token, 0) + 1

        return spam_tokens_freq, not_spam_tokens_freq, total_spam_tokens, total_nonspam_tokens
    

    def compute_likelihoods(self): 
        spam_tokens_freq, not_spam_tokens_freq, total_spam_tokens, total_nonspam_tokens = self.compute_histograms()
        
        spam_likelihoods = {}
        nonspam_likelihoods = {}
        
        for token, count in spam_tokens_freq.items():
            spam_likelihoods[token] = count / total_spam_tokens
        self.spam_likelihoods = spam_likelihoods
        
        for token, count in not_spam_tokens_freq.items():
            nonspam_likelihoods[token] = count / total_nonspam_tokens
        self.nonspam_likelihoods = nonspam_likelihoods
            
        return self.spam_likelihoods, self.nonspam_likelihoods
    
    def fit(self):
        self.compute_initial_probs()
        self.compute_likelihoods()
    
    def predict(self, email: str):
        spam_pred_score = self.spam_prob
        nonspam_pred_score = self.notSpam_prob
        
        spam_likelihoods = self.spam_likelihoods
        nonspam_likelihoods = self.nonspam_likelihoods
        
        for token in email.strip():
            spam_pred_score *= spam_likelihoods.get(token, 0)
            nonspam_pred_score *= nonspam_likelihoods.get(token, 0)
            
            
        if spam_pred_score > nonspam_pred_score:
            return "spam"
        else:
            return "not spam"
            

In [ ]:
nbc = NaiveBayesClassifier()
    
df = pd.read_csv('data/spam_or_not_spam.csv')

nbc.loadData(df)
nbc.fit()
nbc.predict(df['email'][1])

'not spam'